# Business Entity Resolution — Kaggle Runner

This notebook clones your repository and runs the end-to-end **Business Entity Resolution** pipeline on Kaggle.

### Pre-Run Checklist:
1. **Input Data**: Ensure you have attached the challenge dataset using **+ Add Data** (your `student_resource` folder).
2. **Internet Toggle**: Under **Notebook Options** in the right panel, make sure **Internet is ON** (needed for `git clone` & `pip install`).
3. **Accelerator**: `None` (Standard CPU: 4 vCPUs, 30 GB RAM is optimal and fast).

In [ ]:
# Step 1: Install required dependencies
!pip install -q "rapidfuzz>=3.6.0"

In [ ]:
# Step 2: Clone your codebase from GitHub
# Replace this with your repository URL (public repo, or use Personal Access Token for private repo)
# e.g. REPO_URL = "https://github.com/<username>/<repo_name>.git"
REPO_URL = "https://github.com/your-username/your-repo.git"
BRANCH = "main"

from pathlib import Path
import os

working_dir = Path("/kaggle/working")
repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

if (working_dir / repo_name).exists() or (working_dir / "business_entity_resolution").exists():
    print("Repository code already exists in /kaggle/working.")
else:
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    print("Cloning complete.")

In [ ]:
# Step 3: Automatically detect Dataset and Source paths
from pathlib import Path

# 1. Locate dataset in /kaggle/input
dataset_dir = None
for p in Path("/kaggle/input").rglob("train_source1.tsv"):
    dataset_dir = p.parent.parent  # Points to .../student_resource/dataset
    break

if dataset_dir is None:
    raise FileNotFoundError("Could not find train_source1.tsv in /kaggle/input! Did you attach the dataset?")

# 2. Locate run_pipeline.py
src_dir = None
for p in Path("/kaggle/working").rglob("run_pipeline.py"):
    src_dir = p.parent
    break

if src_dir is None:
    raise FileNotFoundError("Could not find run_pipeline.py in /kaggle/working! Please verify the git clone step.")

print("Found dataset directory :", dataset_dir)
print("Found pipeline source   :", src_dir)

In [ ]:
# Step 4: Run the Complete Pipeline (with live unbuffered streaming logs)
# Note: '-u' flag forces real-time unbuffered stdout streaming to Kaggle
# Outputs are saved directly to /kaggle/working/output
!python -u "{src_dir}/run_pipeline.py" run \
    --data-dir "{dataset_dir}" \
    --out-dir /kaggle/working/output \
    --work-dir /kaggle/working/artifacts

In [ ]:
# Step 5: Validate the Generated Submission
validator_script = None
for p in Path("/kaggle/input").rglob("validate_submission.py"):
    validator_script = p
    break

if validator_script and validator_script.exists():
    print(f"Running official validator: {validator_script}")
    !python "{validator_script}" \
        --matching /kaggle/working/output/matching_results.tsv \
        --candidate /kaggle/working/output/candidate_pairs.tsv \
        --test-dir "{dataset_dir}/test"
else:
    print("Running built-in pipeline validator:")
    !python -u "{src_dir}/run_pipeline.py" validate \
        --data-dir "{dataset_dir}" \
        --out-dir /kaggle/working/output

In [ ]:
# Step 6: Preview Output Files
import pandas as pd

matching_path = Path("/kaggle/working/output/matching_results.tsv")
candidate_path = Path("/kaggle/working/output/candidate_pairs.tsv")

print(f"matching_results.tsv size : {matching_path.stat().st_size / 1024:.1f} KB")
print(f"candidate_pairs.tsv size  : {candidate_path.stat().st_size / 1024:.1f} KB")

df_matches = pd.read_csv(matching_path, sep="\t", dtype=str, keep_default_na=False)
print(f"\nTotal test S1 entities: {len(df_matches)}")
has_match = (df_matches["matched_entity_ids"] != "").sum()
print(f"Entities with >= 1 match: {has_match} ({has_match / len(df_matches):.1%})")
print(f"Singletons (no match)   : {len(df_matches) - has_match} ({(len(df_matches) - has_match) / len(df_matches):.1%})")

print("\nFirst 10 predictions:")
display(df_matches.head(10))

In [ ]:
# Step 7: Create the Final Submission Zip Package
import zipfile

TEAM_NAME = "your_team_name"  # <--- Change this to your registered team name!

output_dir = Path("/kaggle/working/output")
zip_path = Path("/kaggle/working") / f"{TEAM_NAME}_submission.zip"

# Locate the business_entity_resolution folder
ber_root = None
for p in Path("/kaggle/working").rglob("ber"):
    if (p / "__init__.py").exists():
        ber_root = p.parent.parent
        break
if not ber_root:
    ber_root = src_dir.parent

print("Packaging submission zip from:", ber_root)
skip_patterns = {"__pycache__", ".ipynb_checkpoints", ".git"}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    # 1. Output files
    z.write(output_dir / "matching_results.tsv", "output/matching_results.tsv")
    z.write(output_dir / "candidate_pairs.tsv", "output/candidate_pairs.tsv")
    
    # 2. Code files under code/business_entity_resolution/
    for f in ber_root.rglob("*"):
        if f.is_file() and not (skip_patterns & set(f.parts)):
            z.write(f, f"code/business_entity_resolution/{f.relative_to(ber_root).as_posix()}")
            
    # 3. Documentation template
    doc = None
    for cand in [dataset_dir.parent / "Documentation_template.md", *Path("/kaggle").rglob("Documentation_template.md")]:
        if cand.exists() and cand.is_file():
            doc = cand
            break
    if doc:
        z.write(doc, "Documentation_template.md")
        print("✓ Included Documentation_template.md")
    else:
        print("Note: Documentation_template.md not found - remember to add it before final upload.")

print(f"\n✓ Submission zip created successfully: {zip_path.name} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
print("Files ready in /kaggle/working/:")
print("- matching_results.tsv (for leaderboard upload)")
print(f"- {zip_path.name} (for final zip submission)")